# LoRA SFT with Taktiny

Fine-tune `HuggingFaceTB/SmolLM2-135M-Instruct` on 100 verified math conversations from `open-r1/OpenThoughts-114k-math`. Decoder parameters are stored in a compact `SeqStack`, while only LoRA adapters on the query and key projections are trained.

In [ ]:
%pip install -q "git+https://github.com/solitarius-ml/taktiny.git@experiment" datasets optax

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
from datasets import load_dataset
from jax.experimental import mesh_utils
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from transformers import AutoTokenizer

from taktiny import (
    DatasetConfig,
    LoraConfig,
    Maestro,
    Takt,
    Trainer,
    TrainingConfig,
    nn,
)
from taktiny.cosettes._common import TransformerContext

In [ ]:
MODEL_REPO = 'HuggingFaceTB/SmolLM2-135M-Instruct'
DATASET_REPO = 'open-r1/OpenThoughts-114k-math'
MAX_SAMPLES = 100
MAX_LENGTH = 256
BATCH_SIZE = jax.device_count()
LEARNING_RATE = 2e-4

devices = mesh_utils.create_device_mesh((jax.device_count(),))
mesh = Mesh(devices, ('fsdp',))
batch_sharding = NamedSharding(mesh, P('fsdp', None))

print(f'JAX devices: {jax.devices()}')
print(f'Global batch size: {BATCH_SIZE}')

## Model and adapters

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = Maestro.from_pretrained(
    MODEL_REPO,
    dtype='bfloat16',
    mesh=mesh,
    compact=True,
)

model = Takt.apply_peft(
    model,
    LoraConfig(
        target_modules=['q_proj', 'k_proj'],
        rank=8,
        alpha=16,
        rngs=nn.Rngs(30),
    ),
)

In [ ]:
parameters = model.flat_parameter_dict()
trainable = {
    name: parameter
    for name, parameter in parameters.items()
    if parameter.trainable
}

print(f'Total parameter tensors: {len(parameters)}')
print(f'Trainable LoRA tensors: {len(trainable)}')
print('\n'.join(trainable))

## Dataset

In [ ]:
def row_messages(row):
    messages = row.get('messages')
    if messages:
        return [
            {'role': message['role'], 'content': message['content']}
            for message in messages
        ]

    role_map = {
        'human': 'user',
        'user': 'user',
        'gpt': 'assistant',
        'assistant': 'assistant',
        'system': 'system',
    }
    messages = [
        {
            'role': role_map[turn['from']],
            'content': turn['value'],
        }
        for turn in row['conversations']
    ]
    if row.get('system') and messages[0]['role'] != 'system':
        messages.insert(0, {'role': 'system', 'content': row['system']})
    return messages


def encode_row(row):
    messages = row_messages(row)
    assistant_index = max(
        index
        for index, message in enumerate(messages)
        if message['role'] == 'assistant'
    )
    prompt_messages = messages[:assistant_index]
    full_messages = messages[:assistant_index + 1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    input_ids = list(encoded['input_ids'])
    labels = [
        token_id if end > len(prompt_text) else -100
        for token_id, (_, end) in zip(
            input_ids,
            encoded['offset_mapping'],
            strict=True,
        )
    ]
    if not any(label != -100 for label in labels):
        return None

    attention_mask = [1] * len(input_ids)
    padding = MAX_LENGTH - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding
    attention_mask += [0] * padding
    labels += [-100] * padding
    return input_ids, attention_mask, labels

In [ ]:
stream = load_dataset(DATASET_REPO, split='train', streaming=True)
stream = stream.filter(lambda row: bool(row['correct']))
stream = stream.shuffle(seed=30, buffer_size=10_000)

samples = []
for row in stream.take(MAX_SAMPLES * 5):
    sample = encode_row(row)
    if sample is not None:
        samples.append(sample)
    if len(samples) == MAX_SAMPLES:
        break

if len(samples) < MAX_SAMPLES:
    raise RuntimeError(f'Only found {len(samples)} usable samples')

print(f'Prepared {len(samples)} samples')

In [ ]:
usable_samples = len(samples) - len(samples) % BATCH_SIZE
batches = []
for start in range(0, usable_samples, BATCH_SIZE):
    chunk = samples[start:start + BATCH_SIZE]
    batches.append(
        (
            np.asarray([sample[0] for sample in chunk], dtype=np.int32),
            np.asarray([sample[1] for sample in chunk], dtype=np.bool_),
            np.asarray([sample[2] for sample in chunk], dtype=np.int32),
        )
    )

print(f'Created {len(batches)} batches with shape {batches[0][0].shape}')

## Assistant-only SFT

In [ ]:
def sft_loss(model, batch):
    input_ids, attention_mask, labels = batch
    causal_context = TransformerContext(
        key_cache=None,
        value_cache=None,
        position_idx=None,
        is_causal=True,
    )
    logits, _ = model(
        input_ids,
        attention_mask=attention_mask[:, None, None, :],
        ctx=causal_context,
    )

    target_ids = labels[:, 1:]
    loss_mask = target_ids != -100
    safe_target_ids = jnp.where(loss_mask, target_ids, 0)
    token_losses = optax.softmax_cross_entropy_with_integer_labels(
        logits[:, :-1, :],
        safe_target_ids,
    )
    loss_mask = loss_mask.astype(token_losses.dtype)
    return jnp.sum(token_losses * loss_mask) / jnp.maximum(
        jnp.sum(loss_mask),
        1,
    )

In [ ]:
first_adapter = next(
    parameter
    for name, parameter in trainable.items()
    if name.endswith('lora_B')
)
adapter_before = np.asarray(jax.device_get(first_adapter.value))
initial_loss = float(sft_loss(model, batches[0]))

trainer = Trainer(
    model,
    loss_fn=sft_loss,
    training_config=TrainingConfig(
        epochs=1,
        max_steps=len(batches),
        learning_rate=LEARNING_RATE,
        log_interval=1,
        jit_compile=True,
    ),
    dataset_config=DatasetConfig(
        dataloader=batches,
        batch_sharding=batch_sharding,
        shuffle=False,
    ),
)

print(f'Initial loss: {initial_loss:.6f}')

In [ ]:
trainer.train()

In [ ]:
final_loss = float(sft_loss(model, batches[0]))
adapter_after = np.asarray(jax.device_get(first_adapter.value))
adapter_changed = not np.array_equal(adapter_before, adapter_after)

print(f'Initial loss: {initial_loss:.6f}')
print(f'Final loss:   {final_loss:.6f}')
print(f'Adapter updated: {adapter_changed}')

if not adapter_changed:
    raise AssertionError('LoRA adapter did not update')
if not np.isfinite(final_loss):
    raise AssertionError('Training produced a non-finite loss')

## Generate with the trained model

In [ ]:
messages = [
    {
        'role': 'user',
        'content': 'If 3x + 7 = 22, find x and explain each step.',
    }
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
input_ids = tokenizer(prompt, return_tensors='np').input_ids
output_ids = model.generate(
    input_ids,
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    key=jax.random.key(30),
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [ ]:
# Optional: save the Taktiny model and tokenizer to the Colab runtime.
# output_dir = '/content/taktiny-smollm2-math-lora'
# model.save_pretrained(output_dir)
# tokenizer.save_pretrained(output_dir)